In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # prevent the silent kernel kill from duplicate OpenMP libs
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib
try: get_ipython().run_line_magic("matplotlib","inline")
except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
try: from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image=None

try: _anchor=Path(__file__).resolve()
except NameError: _anchor=Path.cwd()
for _p in (_anchor,*_anchor.parents):
    if (_p/"project_paths.py").exists(): sys.path.insert(0,str(_p)); break
from project_paths import project_root

ROOT=project_root()
CRUISES=["P45_06","P45_07"]
BIN_DB=2.0
MIN_PER_BIN=3
STEEP=0.5          # uM/m: a bin counts as "stratified" above this |gradient|
OUT_DIR=ROOT/"cruises"/CRUISES[-1]/"_audit"/"suna"; OUT_DIR.mkdir(parents=True,exist_ok=True)

def load_L2(cid):
    f=ROOT/"cruises"/cid/"L2"/"SUNA"/f"{cid}__suna_L2_colocated.csv"
    if not f.exists():
        print(f"  [{cid}] no L2 at {f}\n        run step00 -> step01 -> step02 for {cid} first (just change CRUISE_ID)")
        return None
    df=pd.read_csv(f, parse_dates=["time_utc"])
    for c in ("L1_clean","in_cast"):
        if df[c].dtype!=bool:
            df[c]=df[c].astype(str).str.strip().str.lower().isin(["true","1","1.0"])
    return df

L2={cid:load_L2(cid) for cid in CRUISES}
L2={k:v for k,v in L2.items() if v is not None}
for cid,df in L2.items():
    n=df[(df.in_cast)&(df.L1_clean)].cast.nunique()
    print(f"{cid}: {len(df)} frames, {n} casts with clean in-cast data")

In [ ]:
def cast_bins(df_cast, step=BIN_DB):
    d=df_cast[(df_cast.L1_clean)&(df_cast.pressure_db.notna())].copy()
    if d.empty: return pd.DataFrame()
    d["bin"]=(d.pressure_db//step).astype(int)
    g=d.groupby("bin")
    out=pd.DataFrame({
        "p":g.pressure_db.median(),
        "down":d[d.leg=="down"].groupby("bin").nitrate_uM.median(),
        "up":d[d.leg=="up"].groupby("bin").nitrate_uM.median(),
        "comb":g.nitrate_uM.median(),
        "n":g.size(),
    }).sort_values("p")
    return out

def cruise_pairs(df):
    """Across all casts: (|vertical gradient|, |down-up mismatch|) per usable bin."""
    rows=[]
    for cid,gc in df[(df.in_cast)&(df.L1_clean)].groupby("cast"):
        b=cast_bins(gc)
        if len(b)<3: continue
        p=b.p.to_numpy(); comb=b.comb.to_numpy()
        grad=np.full(len(b),np.nan)
        grad[1:-1]=(comb[2:]-comb[:-2])/(p[2:]-p[:-2])     # central diff, uM per db(~m)
        b=b.assign(grad=np.abs(grad))
        ok=b.down.notna()&b.up.notna()&b.grad.notna()&(b.n>=MIN_PER_BIN)
        for _,r in b[ok].iterrows():
            rows.append(dict(cast=cid,p=r.p,grad=r.grad,mism=abs(r.down-r.up)))
    return pd.DataFrame(rows)

def slope_through_origin(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y); x,y=x[m],y[m]
    if len(x)<3 or np.sum(x**2)==0: return np.nan,np.nan
    s=np.sum(x*y)/np.sum(x**2)
    ss_res=np.sum((y-s*x)**2); ss_tot=np.sum((y-y.mean())**2)
    return s,(1-ss_res/ss_tot if ss_tot>0 else np.nan)

In [ ]:
summary=[]; PAIRS={}
for cid,df in L2.items():
    P=cruise_pairs(df); PAIRS[cid]=P
    steep=P[P.grad>=STEEP]; s,r2=slope_through_origin(P.grad,P.mism)
    summary.append(dict(cruise=cid,
        casts=df[(df.in_cast)&(df.L1_clean)].cast.nunique(), bins=len(P),
        grad_med=round(P.grad.median(),2), grad_p90=round(P.grad.quantile(.9),2),
        grad_max=round(P.grad.max(),2), steep_bins=len(steep),
        mism_steep_med=round(steep.mism.median(),2) if len(steep) else np.nan,
        lag_slope=round(s,2), slope_R2=round(r2,2)))
S=pd.DataFrame(summary)
with pd.option_context("display.width",200): print(S.to_string(index=False))
print("\n  grad_*         how stratified the cruise was (uM/m); low grad_max = gentle gradients")
print("  mism_steep_med down/up disagreement in stratified bins = the lag, in uM")
print("  lag_slope      mismatch per unit gradient = the sensor's lag signature")

if {"P45_06","P45_07"}.issubset(set(S.cruise)):
    s06=float(S[S.cruise=="P45_06"].lag_slope.iloc[0]); s07=float(S[S.cruise=="P45_07"].lag_slope.iloc[0])
    denom=max(abs(s06),abs(s07)); rel=abs(s06-s07)/denom*100 if denom>0 else float("nan")
    print(f"\nVERDICT: lag signatures {s06} (P45_06) vs {s07} (P45_07) differ by {rel:.0f}%.")
    if rel < 25:
        print("  Same instrument behaviour on both cruises: the SUNA always flushed poorly.")
        print("  P45-06 looked fine only because its gradients were gentler and its calibration used one cast.")
    else:
        print("  Flushing differed between cruises: investigate fouling, mounting, or the pump.")
S.to_csv(OUT_DIR/"cruise_hysteresis_comparison.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
pd.concat([P.assign(cruise=cid) for cid, P in PAIRS.items()], ignore_index=True) \
  .to_csv(OUT_DIR / "hysteresis_points.csv", index=False, encoding="utf-8-sig")
print("wrote", OUT_DIR / "hysteresis_points.csv")

In [ ]:
   from matplotlib.figure import Figure
   from matplotlib.backends.backend_agg import FigureCanvasAgg
   f = Figure(); FigureCanvasAgg(f); f.add_subplot(111).plot([0, 1], [0, 1])
   f.savefig(str(OUT_DIR / "_mpltest.png")); print("trivial savefig OK")

In [ ]:
# --- robust comparison figure: no pyplot, no tight_layout ---
import numpy as np, pandas as pd
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

colors = {"P45_06": "#2E6FB0", "P45_07": "#d62728"}
fig = Figure(figsize=(12, 4.6)); FigureCanvasAgg(fig)
ax = fig.subplots(1, 2)

for cid, P in PAIRS.items():
    g = pd.to_numeric(P["grad"], errors="coerce").to_numpy(float)
    m = pd.to_numeric(P["mism"], errors="coerce").to_numpy(float)
    ok = np.isfinite(g) & np.isfinite(m); g, m = g[ok], m[ok]
    c = colors.get(cid, "#888888")
    ax[0].plot(g, m, ".", ms=3, alpha=0.35, color=c, rasterized=True)
    s, r2 = slope_through_origin(g, m)
    if np.isfinite(s) and g.size:
        xx = np.linspace(0, float(np.nanquantile(g, 0.98)), 50)
        ax[0].plot(xx, s*xx, "-", color=c, lw=2, label=f"{cid}: slope {s:.1f} (R2 {r2:.2f})")
ax[0].set_xlabel("|vertical gradient| (uM/m)"); ax[0].set_ylabel("|down - up| (uM)")
ax[0].set_title("Lag signature: hysteresis vs stratification")
ax[0].legend(fontsize=8, frameon=False)

for cid, P in PAIRS.items():
    g = pd.to_numeric(P["grad"], errors="coerce").to_numpy(float); g = g[np.isfinite(g)]
    ax[1].hist(g, bins=np.arange(0, 3.1, 0.15), histtype="step", lw=2,
               color=colors.get(cid, "#888888"), label=cid)
ax[1].axvline(STEEP, color="#999999", ls="--", lw=1)
ax[1].set_xlabel("|vertical gradient| (uM/m)"); ax[1].set_ylabel("bins")
ax[1].set_title("How stratified was each cruise?")
ax[1].legend(fontsize=8, frameon=False)

for a in ax:
    for sp in ("top", "right"): a.spines[sp].set_visible(False)

fig.subplots_adjust(left=0.07, right=0.98, top=0.90, bottom=0.14, wspace=0.22)  # replaces tight_layout
out = OUT_DIR / "cruise_hysteresis_comparison.png"
fig.savefig(out, dpi=135)
print("saved", out)

# safety net: also write the points behind the figure (pure pandas, cannot crash)
try:
    pd.concat([P.assign(cruise=cid) for cid, P in PAIRS.items()],
              ignore_index=True).to_csv(OUT_DIR / "hysteresis_points.csv",
                                        index=False, encoding="utf-8-sig")
    print("saved hysteresis_points.csv")
except Exception as e:
    print("points export skipped:", e)

try:
    from IPython.display import Image, display
    display(Image(filename=str(out)))
except Exception as e:
    print("inline display skipped (PNG is saved):", e)

In [ ]:
# --- detailed plain-language inference + recommendations (all derived from the data) ---
import textwrap
LAG_OK   = 2.0    # lag signature (uM mismatch per uM/m of gradient) at/below which the SUNA
                  # effectively flushes. A clean sensor sits near 0; raise/lower to taste.
R2_TIGHT = 0.40   # above this, the lag-vs-gradient relationship counts as "tight"

Sx = S.set_index("cruise")
def _f(c, col): return float(Sx.loc[c, col])
def status(c):  return "lag-limited" if _f(c, "lag_slope") > LAG_OK else "usable"
cids = list(Sx.index)
sig_list = ", ".join(f"{c} {_f(c,'lag_slope'):.2f}" for c in cids)
bad = [c for c in cids if status(c) == "lag-limited"]
ok  = [c for c in cids if status(c) == "usable"]

def wrap(prefix, body, width=84):
    pad = " " * len(prefix)
    for j, w in enumerate(textwrap.wrap(body, width)):
        print((prefix if j == 0 else pad) + w)

L = "=" * 70
print(L); print("PLAIN-LANGUAGE INFERENCE  (what the numbers mean, and what to do)"); print(L)

print("\nThe sensor's lag, cruise by cruise")
for c in cids:
    print(f"  {c}: for every 1 uM/m of nitrate gradient, the down- and up-cast")
    print(f"         disagree by about {_f(c,'lag_slope'):.1f} uM "
          f"(lag signature {_f(c,'lag_slope'):.2f}, scatter R2 {_f(c,'slope_R2'):.2f})"
          f"  ->  {status(c).upper()}.")

print("\nHow stratified each cruise was")
for c in cids:
    print(f"  {c}: steepest gradient {_f(c,'grad_max'):.2f} uM/m, "
          f"{int(_f(c,'steep_bins'))} steep bins (>= {STEEP} uM/m), "
          f"median gradient {_f(c,'grad_med'):.2f}.")

print("\nWhat the comparison tells us")
if len(cids) >= 2:
    by_strat = sorted(cids, key=lambda c: _f(c, "grad_max"))
    less_s, more_s = by_strat[0], by_strat[-1]
    lv = Sx["lag_slope"].astype(float)
    rel = (lv.max() - lv.min()) / abs(lv.max()) * 100 if lv.max() else float("nan")

    if rel < 25:
        print(f"  - The lag signatures match to {rel:.0f}% ({sig_list}):")
        if not bad:
            wrap("    ", f"the sensor flushes the same way on every cruise, and every one is within "
                         f"tolerance (lag <= {LAG_OK}).")
        elif not ok:
            wrap("    ", "the sensor lags the same way on every cruise - a consistent instrument "
                         "limitation, not a cruise-specific fault.")
        else:
            wrap("    ", "similar magnitudes, but they straddle the tolerance line (see the bottom line).")
    else:
        print(f"  - The lag signatures differ by {rel:.0f}% ({sig_list}):")
        wrap("    ", "the sensor did NOT behave the same on every cruise - something changed between "
                     "them (a deliberate fix, or fouling / mounting / pump differences).")

    if more_s != less_s:
        wrap("  - ", f"{more_s} was the more sharply layered cruise (steepest gradient "
                     f"{_f(more_s,'grad_max'):.2f} vs {_f(less_s,'grad_max'):.2f} uM/m), so the lag had "
                     f"more room to show: its absolute down/up disagreement in stratified water is larger "
                     f"({_f(more_s,'mism_steep_med'):.1f} vs {_f(less_s,'mism_steep_med'):.1f} uM).")
        if status(less_s) == "lag-limited" and _f(less_s, "mism_steep_med") < _f(more_s, "mism_steep_med"):
            wrap("  - ", f"{less_s} looked milder only because its water was gentler, not because the "
                         f"sensor behaved better: per unit gradient its lag ({_f(less_s,'lag_slope'):.2f}) "
                         f"is just as large.")
        elif status(less_s) == "usable":
            wrap("  - ", f"{less_s} genuinely stayed within tolerance, but it was also the least "
                         f"stratified, so that is weak evidence on its own - confirm on a well-stratified "
                         f"cruise.")

    meanr2 = float(Sx["slope_R2"].astype(float).mean())
    if meanr2 < R2_TIGHT:
        wrap("  - ", f"The point-by-point scatter is loose (mean R2 {meanr2:.2f}): the lag is a consistent "
                     f"average tendency, not a tight law - exactly how a flow-cell memory effect behaves "
                     f"(it depends on the water the cell recently held, not just the local gradient).")
    else:
        wrap("  - ", f"The lag scales fairly tightly with gradient (mean R2 {meanr2:.2f}), so the signature "
                     f"is a reliable per-cruise summary of the sensor's behaviour.")
else:
    wrap("  - ", "Only one cruise is loaded; add a second (edit CRUISES in cell 1) to compare behaviour.")

print("\nBottom line")
if not bad:
    wrap("  ", "The SUNA flushes well enough to use its profiles directly on this data; still spot-check "
               "against the bottles at the nitracline.")
elif not ok:
    wrap("  ", "The SUNA flushes poorly here: an instrument limitation, not a one-off. The discrete bottles "
               "are the nitrate structure of record, and the SUNA is a lag-limited companion.")
else:
    wrap("  ", f"Mixed: SUNA profiles are usable on {', '.join(ok)}; the bottles are the record on "
               f"{', '.join(bad)}.")

print("\nRecommendations")
recs = []
if bad:
    recs.append(f"Flag the SUNA on {', '.join(bad)} as lag_limited, and calibrate it only in vertically "
                "homogeneous water where the down- and up-cast agree.")
    recs.append("Use the discrete bottles as the nitrate profile of record wherever the sensor is "
                "lag-limited, and keep a bottle at the sharpest gradient on every cast so the SUNA always "
                "has a reference where it is weakest.")
    recs.append("Do not attempt a response-time (deconvolution) correction to recover the profiles - it was "
                "tested and no time constant reduced the mismatch, so this is near-total cell memory, not a "
                "recoverable first-order lag.")
    recs.append("For future deployments, fix the flushing at source: check the SUNA mounting and "
                "orientation, run the pump if one is fitted, and slow the descent through the nitracline - "
                "then re-run this notebook and confirm the lag signature falls below the tolerance.")
if ok and bad:
    recs.append(f"Compare the deployment logs of {', '.join(ok)} (usable) against {', '.join(bad)} "
                "(lag-limited) to pin down exactly what changed.")
if ok and not bad:
    recs.append("Keep this check in the pipeline so a future flushing problem is caught early, and still "
                "verify the SUNA against the bottles at the nitracline before trusting fine structure.")
for i, r in enumerate(recs, 1):
    wrap(f"  {i}. ", r)
print(L)

In [ ]:
# --- plain-language inference from the comparison ---
Sx = S.set_index("cruise")
def _f(cid, col): return float(Sx.loc[cid, col])

print("="*66)
print("PLAIN-LANGUAGE INFERENCE  (what the numbers above mean)")
print("="*66)

print("\nThe sensor's lag, cruise by cruise")
for cid in Sx.index:
    print(f"  {cid}: for every 1 uM/m of nitrate gradient, the down- and up-cast")
    print(f"         disagree by about {_f(cid,'lag_slope'):.1f} uM "
          f"(lag signature {_f(cid,'lag_slope'):.2f}, scatter R2 {_f(cid,'slope_R2'):.2f}).")

print("\nHow stratified each cruise was")
for cid in Sx.index:
    print(f"  {cid}: steepest gradient {_f(cid,'grad_max'):.2f} uM/m, "
          f"{int(_f(cid,'steep_bins'))} steep bins (>= {STEEP} uM/m), "
          f"median gradient {_f(cid,'grad_med'):.2f}.")

cr = list(Sx.index)
if len(cr) >= 2:
    a, b = cr[0], cr[1]
    la, lb = _f(a,"lag_slope"), _f(b,"lag_slope")
    rel = abs(la-lb)/max(abs(la), abs(lb))*100 if max(abs(la),abs(lb)) else float("nan")
    hi, lo = (a, b) if _f(a,"grad_max") >= _f(b,"grad_max") else (b, a)
    print("\nWhat the comparison tells us")
    print(f"  - The lag signatures ({la:.2f} vs {lb:.2f}) are within {rel:.0f}% of each other:")
    print( "    the same instrument flushing the same way on both cruises, not a cruise-specific fault.")
    print(f"  - {hi} was the more sharply layered cruise (steepest gradient "
          f"{_f(hi,'grad_max'):.2f} vs {_f(lo,'grad_max'):.2f} uM/m), so the lag had more room to show:")
    print(f"    its down/up disagreement in stratified water is larger "
          f"({_f(hi,'mism_steep_med'):.2f} vs {_f(lo,'mism_steep_med'):.2f} uM).")
    print(f"  - {lo} looked acceptable only because its water was gentler, not because the sensor behaved better.")
    print(f"  - The point-by-point scatter is loose (R2 {_f(a,'slope_R2'):.2f} / {_f(b,'slope_R2'):.2f}):")
    print( "    the lag is a consistent average tendency, not a tight law, which is exactly how a flow-cell")
    print( "    memory effect behaves (it depends on recent history, not just the local gradient).")

print("\nBottom line")
print("  The SUNA flushes poorly regardless of cruise: an instrument limitation, not a one-off.")
print("  Use the discrete bottles as the nitrate structure of record, and treat the SUNA profile")
print("  as a lag-limited companion (flagged lag_limited), calibrated only where the water is")
print("  vertically homogeneous and the down/up legs agree.")
print("="*66)